# Laborator: primul generator de text cu LSTM

## Alice in Wonderland, caracter cu caracter

Acest laborator este pentru cineva care abia incepe cu retele neuronale. Nu trebuie sa stii deja ce este un LSTM. Sunt suficiente cunostinte elementare de Python si sa rulezi celulele de sus in jos.

### Ce vei invata
- cum transformam textul in numere;
- cum construim exemple din care modelul invata caracterul urmator;
- ce rol au Embedding, LSTM si stratul de iesire;
- cum se antreneaza si cum citim loss-ul;
- cum putem genera continuari pentru un prompt si compara temperaturile.

### Ideea inainte de cod
Un model de limbaj incearca sa raspunda la intrebarea: „Dupa acest context, ce caracter ar putea urma?”. Pentru `cat`, modelul poate invata perechile `c -> a` si `a -> t`. LSTM citeste secventa in ordine si pastreaza o stare care rezuma contextul anterior. Nu intelege povestea Alice; invata tipare statistice din caracterele textului.

Ruleaza celulele in ordine. Citeste explicatia, ruleaza codul si incearca intrebarile de verificare inainte sa continui.

In [3]:
from pathlib import Path
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Dispozitiv de calcul:", device)

PyTorch: 2.14.0+cpu
Dispozitiv de calcul: cpu


## 1. Incarcam textul

Corpusul este textul folosit ca exemple de invatare. Fisierul `alice_in_wonderland.txt` trebuie sa fie in acelasi folder cu notebookul. Citim tot continutul cu UTF-8, encodarea potrivita pentru text cu litere si simboluri.

In [5]:
TEXT_PATH = Path("alice_in_wonderland.txt")
if not TEXT_PATH.exists():
    raise FileNotFoundError(
        "Nu gasesc alice_in_wonderland.txt. Pune-l langa notebook sau modifica TEXT_PATH."
    )

text = TEXT_PATH.read_text(encoding="utf-8")
if not text:
    raise ValueError("Fisierul este gol.")

print(f"Caractere in text: {len(text):,}")
print("Fragment de verificare:\n")
print(text[:400])

Caractere in text: 148,208
Fragment de verificare:

TITLE: Alice's Adventures in Wonderland
AUTHOR: Lewis Carroll


= CHAPTER I = 
=( Down the Rabbit-Hole )=

  Alice was beginning to get very tired of sitting by her sister
on the bank, and of having nothing to do:  once or twice she had
peeped into the book her sister was reading, but it had no
pictures or conversations in it, `and what is the use of a book,'
thought Alice `without pictures or con


## 2. Construim vocabularul si transformam textul in ID-uri

Vocabularul este multimea caracterelor distincte din carte: litere mari/mici, spatii, punctuatie si randuri noi. Fiecare caracter primeste un numar unic. De exemplu, modelul primeste numarul pentru `A`, nu litera `A` ca text brut.

Pastram doua dictionare: `char2idx` transforma caracterul in numar, iar `idx2char` face operatia inversa. La final verificam ca textul codificat si apoi decodificat este identic cu originalul.

In [6]:
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {char: index for index, char in enumerate(chars)}
idx2char = {index: char for index, char in enumerate(chars)}
encoded_text = np.array([char2idx[char] for char in text], dtype=np.uint8)

decoded_check = "".join(idx2char[int(index)] for index in encoded_text[:80])
assert decoded_check == text[:80]

print("Caractere distincte:", vocab_size)
print("Exemplu de mapare:", list(char2idx.items())[:12])
print("Primele ID-uri:", encoded_text[:20])
print("Decodificarea a reconstruit fragmentul corect.")

Caractere distincte: 81
Exemplu de mapare: [('\n', 0), (' ', 1), ('!', 2), ('"', 3), ("'", 4), ('(', 5), (')', 6), ('*', 7), (',', 8), ('-', 9), ('.', 10), ('0', 11)]
Primele ID-uri: [44 33 44 36 29 20  1 25 66 63 57 59  4 73  1 25 58 76 59 68]
Decodificarea a reconstruit fragmentul corect.


## 3. Din text construim exemple input si target

Modelul invata din ferestre de caractere. Daca luam fragmentul `cat!`, intrarea poate fi `cat`, iar tinta este `at!`: tinta este aceeasi secventa deplasata cu un caracter.

Astfel, in aceeasi secventa, modelul invata la fiecare pozitie caracterul urmator. `seq_length` este contextul vizibil intr-un exemplu. `stride` spune cu cate caractere mutam fereastra pentru urmatorul exemplu; pasul 16 reduce exemplele aproape identice si timpul de rulare.

**Verifica-te:** pentru fragmentul `cat!`, ce caracter trebuie prezis dupa `a`?

In [7]:
seq_length = 64
stride = 16

class CharDataset(Dataset):
    def __init__(self, data, seq_length, stride):
        self.data = data
        self.seq_length = seq_length
        self.stride = stride

    def __len__(self):
        return max(0, (len(self.data) - self.seq_length - 1) // self.stride + 1)

    def __getitem__(self, index):
        start = index * self.stride
        input_ids = self.data[start:start + self.seq_length].astype(np.int64)
        target_ids = self.data[start + 1:start + self.seq_length + 1].astype(np.int64)
        return torch.from_numpy(input_ids), torch.from_numpy(target_ids)

dataset = CharDataset(encoded_text, seq_length, stride)
print("Numar de exemple:", len(dataset))

Numar de exemple: 9259


In [23]:
input_example, target_example = dataset[0]
input_text = "".join(idx2char[int(index)] for index in input_example)
target_text = "".join(idx2char[int(index)] for index in target_example)

print("INPUT :", repr(input_text))
print("TARGET:", repr(target_text))
print("Forma unui input:", tuple(input_example.shape))
print("\nPrimele perechi caracter -> caracter urmator:")
for current_id, next_id in zip(input_example[:12], target_example[:12]):
    print(repr(idx2char[int(current_id)]), "->", repr(idx2char[int(next_id)]))

INPUT : "TITLE: Alice's Adventures in Wonderland\nAUTHOR: Lewis Carroll\n\n\n"
TARGET: "ITLE: Alice's Adventures in Wonderland\nAUTHOR: Lewis Carroll\n\n\n="
Forma unui input: (64,)

Primele perechi caracter -> caracter urmator:
'T' -> 'I'
'I' -> 'T'
'T' -> 'L'
'L' -> 'E'
'E' -> ':'
':' -> ' '
' ' -> 'A'
'A' -> 'l'
'l' -> 'i'
'i' -> 'c'
'c' -> 'e'
'e' -> "'"


## 4. Organizare in batch-uri

Un `batch` este un grup de exemple procesate impreuna. `DataLoader` formeaza grupurile si amesteca ordinea exemplelor de antrenare. `num_workers=0` este ales pentru compatibilitate simpla cu notebookurile Windows.

Forma `(batch, secventa)` inseamna, de exemplu, cateva zeci de fragmente, fiecare cu 64 de ID-uri.

In [9]:
batch_size = 32
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)

input_batch, target_batch = next(iter(loader))
print("Forma batchului input:", tuple(input_batch.shape))
print("Forma batchului target:", tuple(target_batch.shape))
print("Batch-uri intr-o epoca:", len(loader))

Forma batchului input: (32, 64)
Forma batchului target: (32, 64)
Batch-uri intr-o epoca: 289


## 5. Arhitectura LSTM

- **Embedding** transforma fiecare ID de caracter intr-un vector numeric invatabil.
- **LSTM** citeste vectorii in ordine si isi actualizeaza memoria interna. Portile sale invata ce context sa pastreze sau sa uite.
- **Dense** transforma reprezentarea LSTM in cate un scor pentru fiecare caracter din vocabular.

Nu este nevoie sa memorezi formulele. Urmareste formele: intrarea este `(batch, timp)`, embedding-ul adauga o dimensiune de vectori, iar iesirea modelului are o ultima dimensiune egala cu numarul de caractere posibile.

In [17]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, hidden=None):
        embedded = self.embedding(input_ids)
        sequence_output, hidden = self.lstm(embedded, hidden)
        logits = self.output_layer(sequence_output)
        return logits, hidden

model = CharLSTM(vocab_size).to(device)
print(model)

CharLSTM(
  (embedding): Embedding(81, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.2)
  (output_layer): Linear(in_features=128, out_features=81, bias=True)
)


In [18]:
with torch.no_grad():
    example_logits, _ = model(input_batch.to(device))

print("Input:", tuple(input_batch.shape))
print("Scoruri produse de model:", tuple(example_logits.shape))
print("Ultima dimensiune este vocab_size:", vocab_size)

Input: (32, 64)
Scoruri produse de model: (32, 64, 81)
Ultima dimensiune este vocab_size: 81


## 6. Scor, loss si invatare

Modelul produce un scor (logit) pentru fiecare caracter posibil, la fiecare pozitie din secventa. `CrossEntropyLoss` compara scorurile cu caracterele tinta si da o valoare numita loss. Cu cat loss-ul este mai mic, cu atat modelul a fost mai potrivit pentru exemplele analizate.

Optimizatorul Adam modifica ponderile modelului pentru a reduce loss-ul. O epoca este o trecere prin toate batch-urile de antrenare. Afisam loss-ul de antrenare; nu este un rezultat de validare si nu dovedeste singur ca modelul generalizeaza.

In [19]:
num_epochs = 5
learning_rate = 0.001
checkpoint_path = Path("alice_lstm_checkpoint.pt")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

input_on_device = input_batch.to(device)
target_on_device = target_batch.to(device)
with torch.no_grad():
    initial_logits, _ = model(input_on_device)
    initial_loss = criterion(
        initial_logits.reshape(-1, vocab_size),
        target_on_device.reshape(-1),
    )

print("Loss pe un batch inainte de antrenare/incarcarea checkpoint-ului:", round(initial_loss.item(), 4))
print("Epoci maxime:", num_epochs)
print("Rata de invatare:", learning_rate)
print("Functia de pierdere:", criterion)

Loss pe un batch inainte de antrenare/incarcarea checkpoint-ului: 4.396
Epoci maxime: 5
Rata de invatare: 0.001
Functia de pierdere: CrossEntropyLoss()


## 7. Antrenam si salvam modelul

La fiecare batch: modelul calculeaza scoruri, loss-ul masoara eroarea, `backward()` calculeaza cum trebuie ajustate ponderile, iar Adam le actualizeaza. Gradient clipping limiteaza pasii foarte mari si poate stabiliza antrenarea.

La finalul fiecarei epoci salvam un checkpoint cu ponderile, optimizatorul si vocabularul. Daca rulezi din nou notebookul cu acelasi text si aceiasi parametri, incercam sa reluam de la checkpoint in loc sa antrenam iar de la zero.

### Rezultatul unei antrenari pe corpusul Alice

In rularea care a creat checkpoint-ul din folder, loss-ul mediu de antrenare a fost:

| Epoca | Loss |
|---:|---:|
| 1 | 2.7768 |
| 2 | 2.0984 |
| 3 | 1.8719 |
| 4 | 1.7435 |
| 5 | 1.6555 |

Loss-ul a scazut in acea rulare. Este loss de antrenare, nu de validare; alte initializari sau modificari ale setarilor pot produce valori diferite.

In [20]:
start_epoch = 1
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    same_data = checkpoint["vocab"] == chars
    same_sequence = checkpoint["seq_length"] == seq_length
    if same_data and same_sequence:
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        start_epoch = checkpoint["epoch"] + 1
        print(f"Checkpoint compatibil gasit: {start_epoch - 1} epoci deja antrenate.")
    else:
        print("Checkpoint incompatibil; antrenam un model nou.")

if start_epoch <= num_epochs:
    print(f"Antrenare pe {device}; {len(loader)} batch-uri per epoca.")
    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        total_loss = 0.0
        start_time = time.time()

        for input_batch, target_batch in loader:
            input_batch = input_batch.to(device, non_blocking=True)
            target_batch = target_batch.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits, _ = model(input_batch)
            loss = criterion(logits.reshape(-1, vocab_size), target_batch.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()

        average_loss = total_loss / len(loader)
        elapsed = time.time() - start_time
        print(f"Epoca {epoch}/{num_epochs} | loss: {average_loss:.4f} | timp: {elapsed:.1f}s")
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "vocab": chars,
                "seq_length": seq_length,
            },
            checkpoint_path,
        )
else:
    print(f"Checkpoint-ul este deja antrenat pana la epoca {start_epoch - 1}.")

Checkpoint compatibil gasit: 5 epoci deja antrenate.
Checkpoint-ul este deja antrenat pana la epoca 5.


## 8. Generam caractere si comparam temperatura

Dupa antrenare, modelul poate continua un prompt. La fiecare pas luam scorurile pentru ultimul caracter si le transformam in probabilitati. Apoi alegem aleatoriu un caracter, il adaugam la context si repetam.

`temperature` modifica distributia inainte de alegere:
- valoare mica: alegerile probabile devin dominante, deci textul tinde sa fie mai previzibil;
- valoare mare: si caracterele cu scoruri mai mici pot fi alese, deci creste varietatea si riscul de forme incoerente.

Temperatura nu modifica ponderile si nu antreneaza din nou modelul.

In [21]:
def generate_text(model, seed_text, length=120, temperature=0.7):
    if temperature <= 0:
        raise ValueError("temperature trebuie sa fie mai mare decat zero.")
    if not seed_text:
        raise ValueError("Promptul trebuie sa contina cel putin un caracter.")

    unknown_chars = sorted(set(seed_text) - set(char2idx))
    if unknown_chars:
        raise ValueError(f"Caractere absente din vocabular: {unknown_chars!r}")

    model.eval()
    input_ids = torch.tensor(
        [[char2idx[char] for char in seed_text]],
        dtype=torch.long,
        device=device,
    )
    generated = seed_text

    with torch.no_grad():
        logits, hidden = model(input_ids)
        for step in range(length):
            probabilities = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            generated += idx2char[next_id.item()]
            if step + 1 < length:
                logits, hidden = model(next_id, hidden)

    return generated

example_prompt = "Alice was "
for temperature in (0.4, 0.8, 1.2):
    print(f"\n--- Temperature = {temperature} ---")
    print(generate_text(model, example_prompt, length=160, temperature=temperature))


--- Temperature = 0.4 ---
Alice was been to say and corsed in the was and her one the King and you know and one the see the say the much a pancing to the Mock of the Caterping to she was she did w

--- Temperature = 0.8 ---
Alice was not to her, surn'tther sat it
down to had mear, I's more mine was as soly she celden!
thing the bast, and one had to his to anch its went?' said teid was so dol

--- Temperature = 1.2 ---
Alice was too!  Alice' `And what, and lookedsed; whink
she-polest of
courtan avion'ting sold imshk lanby "Tem labby-
`They feet hat all surning, quioking nook of othen's 


## 9. Practica: ce cuvant ar putea urma?

Celula urmatoare porneste de la un prompt si eșantioneaza mai multe continuari. Rezultatele sunt **mostre**: modelul lucreaza la nivel de caracter, iar functia afiseaza caracterele pana la primul spatiu sau semn de punctuatie. Nu este o lista de cuvinte garantate si nici nu intelege sensul propozitiei.

Schimba `prompt` in celula urmatoare, de exemplu in `The rabbit ` sau `Alice was `, si ruleaz-o din nou. Promptul trebuie sa foloseasca doar caractere prezente in corpus.

**Noteaza observatiile:** ce se schimba intre temperaturile 0.4, 0.8 si 1.2? La care apar cele mai multe forme cunoscute? La care apare cea mai mare varietate?

In [22]:
def suggest_next_words(
    prompt,
    temperatures=(0.4, 0.8, 1.2),
    num_candidates=4,
    chars_per_candidate=48,
):
    if not prompt.strip():
        raise ValueError("Scrie un prompt in variabila prompt.")
    if num_candidates < 1:
        raise ValueError("num_candidates trebuie sa fie cel putin 1.")

    context = prompt if prompt[-1].isspace() else prompt + " "
    unknown_chars = sorted(set(context) - set(char2idx))
    if unknown_chars:
        raise ValueError(f"Caractere absente din vocabular: {unknown_chars!r}")

    for temperature in temperatures:
        if temperature <= 0:
            raise ValueError("Toate temperaturile trebuie sa fie mai mari decat 0.")

        candidates = []
        attempts = 0
        while len(candidates) < num_candidates and attempts < num_candidates * 5:
            attempts += 1
            sample = generate_text(
                model,
                context,
                length=chars_per_candidate,
                temperature=temperature,
            )
            continuation = sample[len(context):]
            word_chars = []
            for character in continuation:
                if character.isspace() or character in ",.!?;:()[]{}":
                    break
                word_chars.append(character)

            candidate = "".join(word_chars).strip("'\"`-")
            if candidate and candidate not in candidates:
                candidates.append(candidate)

        print(f"\nTemperature = {temperature}")
        print("Cuvinte urmatoare esantionate:", ", ".join(candidates))


# Schimba textul dintre ghilimele si ruleaza din nou doar aceasta celula.
prompt = "Alice was "
suggest_next_words(prompt)


Temperature = 0.4
Cuvinte urmatoare esantionate: the, in, mane, she

Temperature = 0.8
Cuvinte urmatoare esantionate: the, ong, poor, be

Temperature = 1.2
Cuvinte urmatoare esantionate: as, it, bake, want
